# Experiment Power Analysis under Partial Compliance

## Background
To design an A/B test experiment, it is a good idea to understand the conditions required to achieve statistical significance of the effect of the intervention.

We might declare a treatment not effective when in reality is effective (type II error) but we did not have the adequate conditions to see a significant effect on the metric of interest.

# The Power Analysis Approach here
In this note, we are building on top of the LATE estimation using Bayesian Gibbs sampling approach. Compared to a standard normal approximation in a t-test or difference in means, we need to integrate the variability of the non-exposed population and their conversion rate. 

In the current approach, we: 
1. Simulatete the __average__ user counts given a set of parameters
2. Run the LATE estimation based on Gibbss Sampling
3. Find lower bound of the posterior credible interval. If the bound is over zero, the lift can be detected
4. Peform binary search to find the minimum percentage of exposed users in the treatment group

We expect a lower estimation power, i.e. we need more exposed users when we need to infer the conversion rate in the control group for the __never-targeted__ users.

The functions have been packed in a module for easier use.

The functions here assume the module __late_estimators__ which include the LATE gibbs sampling estimator, and other functions needed for the simulation. To replicate the results, please copy the folder **effectEstimators**)

In [5]:
import numpy as np
from effectEstimators import late_estimators as late

import matplotlib
import matplotlib.pyplot as plt

Function to evaluate the credible intervals given a set of parameters. Parameters are
* __pd_in__: Probability of user exposure
* __hold_out__: Control proportion hold out
* __p0_d0__: Probability of conversion for non-exposed (never-targeted) users
* __p0_d1__: Baseline probability of conversion for exposed users
* __lift_d1__: Conversion rate lift proportion for the exposed users
* __n_total__: Total number of users to be potentially exposed
* __test_sig__: Significance of the test for the credible interval estimation

In [6]:
# Power estimation analysis evaluation to find lower confidence interval bound to be over zero
# It returns 1 if the lower confidence interval in LATE or LATE lift is greater than zero
def f_sig_eval(pd_in, hold_out, p0_d0, p0_d1, lift_d1, n_total, test_sig):
    # priors
    alpha_0 = 0.50
    beta_0 = 0.50
    # Iterations
    n_burnin = 5000
    n_samples = 100000

    user_counts = late.count_point_est(1 - hold_out,
                                       pd_in,
                                       p0_d0,
                                       p0_d1,
                                       (1 + lift_d1) * p0_d1,
                                       n_total)
    [theta_d1_samples, theta_d0_samples, theta_n_samples, p_sel_samples] = late.effect_mcmcEst(user_counts, n_burnin,
                                                                                               n_samples, alpha_0,
                                                                                               beta_0)
    late_samp = (theta_d1_samples - theta_d0_samples)
    lift_samp = late_samp / theta_d0_samples
    lift_low_ci = np.percentile(lift_samp, 100 * test_sig / 2)
    late_low_ci = np.percentile(late_samp, 100 * test_sig / 2)

    if lift_low_ci > 0 or late_low_ci > 0:
        return 1
    else:
        return 0

Binary Search to find the minimum probability of user exposure given all the other parameters. It could be performed for other metrics of interest.

In [7]:
# Performs binary search to find the minimum p_sel (probability of targeting)
def search_psel(array, hold_out, p0_d0, p0_d1, lift_d1, n_total, test_sig):
    lower = 0
    upper = len(array) - 1
    upper_val = f_sig_eval(array[upper], hold_out, p0_d0, p0_d1, lift_d1, n_total, test_sig)
    lower_val = f_sig_eval(array[lower], hold_out, p0_d0, p0_d1, lift_d1, n_total, test_sig)

    if upper_val == 0:
        return array[upper], 0.0
    elif lower_val == 1:
        return array[lower], 1.0

    while (upper - lower) > 1:
        x = lower + (upper - lower) // 2
        val = f_sig_eval(array[x], hold_out, p0_d0, p0_d1, lift_d1, n_total, test_sig)
        if val == 0:
            lower = x
        elif val == 1:
            upper = x
        #print([array[lower], array[upper], val])
    return array[upper], 1.0


Now some testing to replicate the results at the power analysis with Ghost Ads 

In [8]:
# Parameters

hold_out = 0.5
p0_d1 = 4.7e-5
p0_d0 = 3e-7
lift_d1 = 0.10
p_sel = 0.3
test_sig = 0.1
n_total = 100e6

# hold increases and p_sel increases
inc_hold = 0.02
inc_sel = 0.01

p_sel_vals = [(x+1)*inc_sel for x in range(int(0.98/inc_sel))]
hold_out_vals = [(x+1)*inc_hold for x in range(int(0.5/inc_hold))]
n_total_vals = [20e6, 30e6, 40e6, 50e6, 60e6, 70e6, 80e6, 90e6, 100e6]


## Testing for different hold-outs
p_sel_sig = np.zeros(len(hold_out_vals))
sig_array = np.zeros(len(hold_out_vals))
i = 0
for hold_out in hold_out_vals:
    [p_sel_sig[i], sig_array[i]] = search_psel(p_sel_vals, hold_out, p0_d0, p0_d1, lift_d1, n_total, test_sig)
    print(p_sel_sig[i], sig_array[i], hold_out)
    i += 1


0.98 0.0 0.02
0.98 0.0 0.04
0.98 0.0 0.06
0.85 1.0 0.08
0.7000000000000001 1.0 0.1
0.6 1.0 0.12
0.53 1.0 0.14
0.47000000000000003 1.0 0.16
0.43 1.0 0.18
0.4 1.0 0.2
0.37 1.0 0.22
0.35000000000000003 1.0 0.24
0.33 1.0 0.26
0.31 1.0 0.28
0.3 1.0 0.3
0.29 1.0 0.32
0.27 1.0 0.34
0.26 1.0 0.36
0.26 1.0 0.38
0.25 1.0 0.4
0.25 1.0 0.42
0.25 1.0 0.44
0.24 1.0 0.46
0.23 1.0 0.48
0.24 1.0 0.5


We need a hold-out of least 8% of the total population. That hold-out leads to 85% of the total population that need to be exposed of 100 Million total users for this particular case. 